# Step 1: Instrument Signature Removal (ISR)

ISR transforms a **raw CCD frame** into a clean image where the pixel values
are proportional to the incoming photon flux. Every instrumental artifact
must be removed before we can measure galaxy shapes.

**LSST task:** `lsst.ip.isr.IsrTask`  
**Input:** `raw` exposure + calibration products  
**Output:** `postISRCCD`

**Reference:** Bosch et al. (2018) §3.1

## What ISR Corrects

Each correction addresses a specific detector/optics artifact:

### 1.1 Overscan Subtraction
CCDs have extra columns (the "overscan region") that aren't illuminated.
These record the **bias level** — the electronic zero-point that drifts
with temperature and time. ISR fits and subtracts this row-by-row.

```
raw_pixel = true_signal + bias_level + dark_current + flat_response + noise
```

### 1.2 Bias Subtraction
After removing the overscan, a **master bias frame** (median of many
zero-second exposures) is subtracted to remove the 2D spatial structure
of the readout electronics.

### 1.3 Dark Current Subtraction
Thermal electrons accumulate even without light. A **master dark frame**
(long exposure with shutter closed), scaled to the science exposure time,
is subtracted. Modern CCDs cooled to ~−100°C have very low dark current,
but it still matters for long exposures.

### 1.4 Flat-fielding
Pixel-to-pixel sensitivity variations (due to manufacturing, QE differences,
vignetting, dust) are corrected by dividing by a **master flat** — a
uniformly-illuminated exposure.

$$I_{\text{corrected}} = \frac{I_{\text{raw}} - \text{bias} - \text{dark}}{\text{flat}}$$

### 1.5 Crosstalk Correction
Electronic coupling between amplifiers causes a ghost of one amp's signal
to appear in another. This is modeled as a linear matrix and subtracted.
Particularly important for HSC's 4-amplifier readout per CCD.

### 1.6 Brighter-Fatter Correction
**Critical for shape measurement.** In thick, fully-depleted CCDs (like
those in HSC and LSST), charge accumulated in a pixel modifies the
local electric field, causing subsequently-arriving photons to land in
neighboring pixels. Bright stars and galaxy cores appear **fatter** than
they should.

The correction:
1. Estimate the pixel-to-pixel charge redistribution kernel from flat-field
   noise correlations (assumes curl-free E-field)
2. Apply the inverse kernel to deconvolve the effect

Without this, PSF models derived from bright stars would be systematically
broader than the true PSF, leading to **under-correction** of galaxy shapes.

### 1.7 Additional Corrections
- **Linearity:** CCD response deviates from linear at high counts
- **Fringe removal:** Thin-film interference in red/NIR bands (i, z, y)
- **Bad pixel masking:** Flag known dead/hot pixels
- **Cosmic ray detection:** Identify and mask CR hits (preliminary)

## Why ISR Matters for Weak Lensing

Shape measurement requires that the pixel values faithfully represent the
sky brightness convolved with the PSF — nothing more. Any residual
instrumental signature biases shapes:

| Artifact | Effect on shapes |
|----------|------------------|
| Residual flat-field errors | Spatially-varying flux bias → selection effects |
| Brighter-fatter effect | Broadens bright sources → PSF model mismatch |
| Crosstalk ghosts | Phantom sources with correlated shapes |
| Bad pixels | Missing data near galaxy centers → biased moments |
| Fringe residuals | Coherent flux pattern → spurious detections |

## The ISR Task in the Pipeline

```python
from lsst.ip.isr import IsrTask

# The task is configured with a Config object
config = IsrTask.ConfigClass()

# Key configuration options:
config.doBias = True          # subtract master bias
config.doDark = True          # subtract master dark
config.doFlat = True          # divide by master flat
config.doCrosstalk = True     # correct amplifier crosstalk
config.doBrighterFatter = True  # apply brighter-fatter correction
config.doFringe = True        # remove fringe pattern
config.doLinearize = True     # correct CCD non-linearity
config.doDefect = True        # mask known bad pixels

# Create and run
isr_task = IsrTask(config=config)
result = isr_task.run(
    ccdExposure=raw_exposure,
    bias=master_bias,
    dark=master_dark,
    flat=master_flat,
    # ... other calibration products
)
postISRCCD = result.exposure
```

## Inspecting ISR Outputs

```python
# Via the Butler (after pipeline has run):
postISR = butler.get('postISRCCD', visit=903334, detector=16)

# The image, variance, and mask planes
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Science image
im = postISR.image.array
vmin, vmax = np.percentile(im, [1, 99])
axes[0].imshow(im, vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
axes[0].set_title('Image (post-ISR)')

# Variance plane
axes[1].imshow(np.sqrt(postISR.variance.array), cmap='magma', origin='lower')
axes[1].set_title('Noise (sqrt variance)')

# Mask plane
axes[2].imshow(postISR.mask.array, cmap='tab20', origin='lower')
axes[2].set_title('Mask (bad pixels, CRs, etc.)')

plt.tight_layout()
```

### Mask bit definitions
```python
# Each bit in the mask plane flags a different condition
for name, bit in postISR.mask.getMaskPlaneDict().items():
    print(f'{name:20s} bit {bit}')
```

Common flags: `BAD`, `SAT` (saturated), `CR` (cosmic ray), `EDGE`,
`DETECTED`, `INTRP` (interpolated), `NO_DATA`.

In [ ]:
# --- Simulation stand-in ---
# Since we may not have the LSST stack installed, let's simulate
# what ISR does using basic numpy/astropy to build intuition.

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
ny, nx = 512, 512

# 1. Create a "true sky" image with a few galaxies
true_sky = np.zeros((ny, nx))
# Add some Gaussian "galaxies"
yy, xx = np.mgrid[:ny, :nx]
for _ in range(15):
    cx, cy = rng.integers(50, nx-50), rng.integers(50, ny-50)
    flux = 10**rng.uniform(3, 5)
    sigma = rng.uniform(2, 8)
    true_sky += flux * np.exp(-((xx-cx)**2 + (yy-cy)**2) / (2*sigma**2)) / (2*np.pi*sigma**2)

# 2. Simulate instrumental effects
bias_level = 1000.0  # electrons
bias_2d = bias_level + 5 * rng.normal(size=(ny, nx))  # spatial structure
dark_rate = 0.01  # e/s/pixel
exposure_time = 300  # seconds
dark = dark_rate * exposure_time + 0.5 * rng.normal(size=(ny, nx))
flat = 1.0 + 0.05 * np.sin(2*np.pi*xx/nx) * np.cos(2*np.pi*yy/ny)  # ~5% variation
flat += 0.01 * rng.normal(size=(ny, nx))  # pixel-to-pixel

# 3. Raw image = (sky * flat) + dark + bias + noise
raw = (true_sky * flat) + dark + bias_2d
raw += np.sqrt(np.maximum(raw, 0)) * rng.normal(size=(ny, nx))  # Poisson-like noise

# 4. Apply ISR corrections
step1 = raw - bias_2d         # bias subtraction
step2 = step1 - dark          # dark subtraction
step3 = step2 / flat          # flat fielding

print(f"Raw image:       mean={raw.mean():.1f}, std={raw.std():.1f}")
print(f"After bias sub:  mean={step1.mean():.1f}, std={step1.std():.1f}")
print(f"After dark sub:  mean={step2.mean():.1f}, std={step2.std():.1f}")
print(f"After flat div:  mean={step3.mean():.1f}, std={step3.std():.1f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: calibration frames
axes[0, 0].imshow(bias_2d, cmap='gray', origin='lower')
axes[0, 0].set_title('Master Bias')
axes[0, 1].imshow(dark, cmap='gray', origin='lower')
axes[0, 1].set_title('Master Dark')
axes[0, 2].imshow(flat, cmap='RdBu_r', origin='lower', vmin=0.9, vmax=1.1)
axes[0, 2].set_title('Master Flat')

# Row 2: processing
vmin, vmax = np.percentile(true_sky, [1, 99.5])
axes[1, 0].imshow(raw, cmap='gray', origin='lower')
axes[1, 0].set_title('Raw (sky + bias + dark + flat effects)')
axes[1, 1].imshow(step3, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[1, 1].set_title('After ISR')
axes[1, 2].imshow(true_sky, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[1, 2].set_title('True Sky (ground truth)')

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Instrument Signature Removal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/isr_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residual after ISR
residual = step3 - true_sky

fig, ax = plt.subplots(figsize=(6, 5))
ax.hist(residual.ravel(), bins=100, range=(-50, 50), color='steelblue',
        edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', ls='--', lw=2)
ax.set_xlabel('Residual (corrected - true)')
ax.set_ylabel('Pixel count')
ax.set_title(f'ISR Residual: mean={residual.mean():.2f}, std={residual.std():.2f}')
plt.tight_layout()
plt.show()

print("ISR should leave only sky noise + photon noise.")
print("Any systematic structure in the residual indicates imperfect calibration.")

## Summary

| What | Before ISR | After ISR |
|------|-----------|----------|
| Pixel values | Arbitrary (bias + dark + flat × sky) | Proportional to sky flux |
| Noise model | Unknown | Poisson (sky+source) + readnoise |
| Spatial artifacts | Flat-field, fringes, crosstalk | Removed (to calibration accuracy) |
| Charge effects | Brighter-fatter distortion present | Corrected |

**Next:** [03_characterization.ipynb](03_characterization.ipynb) — Background estimation and PSF modeling